<a href="https://colab.research.google.com/github/usama488/bioinformatics-analysis/blob/main/Multi_Omics_Integration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#  Multi-Omics Data Integration

**Project 10  Bioinformatics

Is notebook mein hum **teen alag omics layers** — **Genomics (mutations)**, **Transcriptomics (gene expression)**, aur **Proteomics (protein abundance)** — ko integrate kar ke ek unified patient-level model banain gay, jo sirf ek omics layer se zyada powerful predictions deta hai.

**Pipeline:**
## 1. Teen omics datasets simulate karna (same patients ke liye)
# 2. Har layer ka independent analysis (EDA)
# 3. Data integration (concatenation-based multi-omics fusion)
# 4. Joint dimensionality reduction (PCA) — multi-omics ko ek space mein dekhna
# 5. Model comparison — single-omics vs multi-omics prediction power
# 6. **Runtime cell** — ek patient ke teenon omics values daal kar direct predict karein


## 1. Setup & Imports

In [1]:
# !pip install -q plotly scikit-learn pandas numpy ipywidgets

import numpy as np
import pandas as pd

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix

import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

np.random.seed(42)


## 2. Simulate Three Omics Layers (Same Patient Cohort)

Har patient ke liye **teen alag data types** generate kar rahe hain:
-  **Genomics**: mutation counts in 8 cancer-related genes (0/1 mutated)
-  **Transcriptomics**: expression levels of 15 genes
-  **Proteomics**: abundance of 10 proteins

Disease status (**Cancer / No Cancer**) har layer mein partially reflect hota hai — is liye akela ek layer poori tasveer nahi deta, integration se accuracy behtar hoti hai.


In [2]:
n_patients = 300
rng = np.random.default_rng(42)

disease_status = rng.choice(["Cancer", "No Cancer"], size=n_patients, p=[0.45, 0.55])
patient_ids = [f"P{str(i+1).zfill(4)}" for i in range(n_patients)]

# --- Genomics layer: mutation presence (0/1) in cancer-related genes ---
cancer_genes = ["TP53", "BRCA1", "BRCA2", "KRAS", "EGFR", "PTEN", "APC", "MYC"]
genomics_data = {}
for gene in cancer_genes:
    mut_prob = rng.uniform(0.35, 0.6)
    genomics_data[gene] = np.where(
        disease_status == "Cancer",
        rng.binomial(1, mut_prob, n_patients),
        rng.binomial(1, mut_prob * 0.25, n_patients)
    )
genomics_df = pd.DataFrame(genomics_data, index=patient_ids)

# --- Transcriptomics layer: gene expression (log2 scale) ---
expr_genes = [f"EXPR_{i}" for i in range(1, 16)]
transcriptomics_data = {}
for i, gene in enumerate(expr_genes):
    shift = rng.uniform(0.3, 1.8) if i % 3 == 0 else rng.uniform(-0.2, 0.2)
    base = rng.normal(6, 1.0, n_patients)
    transcriptomics_data[gene] = (np.where(disease_status == "Cancer", base + shift, base)).round(2)
transcriptomics_df = pd.DataFrame(transcriptomics_data, index=patient_ids)

# --- Proteomics layer: protein abundance (normalized intensity) ---
proteins = [f"PROT_{i}" for i in range(1, 11)]
proteomics_data = {}
for i, prot in enumerate(proteins):
    shift = rng.uniform(0.4, 1.5) if i % 2 == 0 else rng.uniform(-0.3, 0.3)
    base = rng.normal(4, 0.8, n_patients)
    proteomics_data[prot] = (np.where(disease_status == "Cancer", base + shift, base)).round(2)
proteomics_df = pd.DataFrame(proteomics_data, index=patient_ids)

clinical_df = pd.DataFrame({"disease_status": disease_status}, index=patient_ids)

print(f"Genomics: {genomics_df.shape}  |  Transcriptomics: {transcriptomics_df.shape}  |  Proteomics: {proteomics_df.shape}")
genomics_df.head(3)


Genomics: (300, 8)  |  Transcriptomics: (300, 15)  |  Proteomics: (300, 10)


,TP53,BRCA1,BRCA2,KRAS,EGFR,PTEN,APC,MYC
P0001,0,0,0,0,0,0,0,0
P0002,1,0,1,0,0,0,1,0
P0003,0,0,0,0,0,0,0,0


## 3. Independent Layer-by-Layer Exploration

In [3]:
fig = make_subplots(rows=1, cols=3, subplot_titles=("Genomics: Mutation Rate", "Transcriptomics: Sample", "Proteomics: Sample"))

mut_rate = genomics_df.groupby(disease_status).mean().T
for status in ["Cancer", "No Cancer"]:
    fig.add_trace(go.Bar(x=mut_rate.index, y=mut_rate[status], name=status,
                          marker_color="#E63946" if status == "Cancer" else "#2E86AB"), row=1, col=1)

for status in ["Cancer", "No Cancer"]:
    vals = transcriptomics_df.loc[disease_status == status, "EXPR_1"]
    fig.add_trace(go.Box(y=vals, name=status, marker_color="#E63946" if status == "Cancer" else "#2E86AB", showlegend=False), row=1, col=2)

for status in ["Cancer", "No Cancer"]:
    vals = proteomics_df.loc[disease_status == status, "PROT_1"]
    fig.add_trace(go.Box(y=vals, name=status, marker_color="#E63946" if status == "Cancer" else "#2E86AB", showlegend=False), row=1, col=3)

fig.update_layout(height=450, title_text="Quick Look at Each Omics Layer", barmode='group')
fig.show()


In [4]:
fig = px.imshow(
    genomics_df.values.T, x=patient_ids, y=cancer_genes, color_continuous_scale="Reds", aspect="auto",
    title="Genomics Layer — Mutation Landscape (subset of patients)",
    labels=dict(color="Mutated")
)
fig.update_xaxes(showticklabels=False)
fig.update_layout(height=400)
fig.show()


## 4. Data Integration — Combine All Omics Layers

In [5]:
# Concatenation-based integration: align all layers on patient_id, combine into one feature matrix
integrated_df = pd.concat([genomics_df, transcriptomics_df, proteomics_df], axis=1)
integrated_df["disease_status"] = clinical_df["disease_status"]

print(f"Integrated dataset shape: {integrated_df.shape}")
print(f"  → {genomics_df.shape[1]} genomic + {transcriptomics_df.shape[1]} transcriptomic + {proteomics_df.shape[1]} proteomic features")
integrated_df.head(3)


Integrated dataset shape: (300, 34)
  → 8 genomic + 15 transcriptomic + 10 proteomic features


,TP53,BRCA1,BRCA2,KRAS,EGFR,PTEN,APC,MYC,EXPR_1,EXPR_2,...,PROT_2,PROT_3,PROT_4,PROT_5,PROT_6,PROT_7,PROT_8,PROT_9,PROT_10,disease_status
P0001,0,0,0,0,0,0,0,0,5.51,7.77,...,2.50,4.24,3.70,2.29,3.64,3.12,4.53,2.73,3.50,No Cancer
P0002,1,0,1,0,0,0,1,0,7.89,6.57,...,4.04,3.95,5.18,4.07,3.44,5.23,3.02,4.49,3.35,Cancer
P0003,0,0,0,0,0,0,0,0,4.44,5.10,...,2.82,4.13,4.11,4.44,4.25,4.36,3.76,2.35,4.09,No Cancer


## 5. Joint Dimensionality Reduction (PCA on Integrated Data)

In [6]:
feature_cols_all = cancer_genes + expr_genes + proteins
X_all = integrated_df[feature_cols_all]

pca = PCA(n_components=2)
pcs = pca.fit_transform(StandardScaler().fit_transform(X_all))
pca_df = pd.DataFrame(pcs, columns=["PC1", "PC2"])
pca_df["disease_status"] = disease_status
pca_df["source"] = "Multi-Omics (Integrated)"

fig = px.scatter(
    pca_df, x="PC1", y="PC2", color="disease_status",
    title=f"PCA of Integrated Multi-Omics Data (PC1: {pca.explained_variance_ratio_[0]*100:.1f}%, PC2: {pca.explained_variance_ratio_[1]*100:.1f}%)",
    template="plotly_white", color_discrete_map={"Cancer": "#E63946", "No Cancer": "#2E86AB"}
)
fig.update_traces(marker=dict(size=9, line=dict(width=0.5, color='white')))
fig.update_layout(height=550)
fig.show()


## 6. Model Comparison — Single-Omics vs Multi-Omics

In [7]:
y = (disease_status == "Cancer").astype(int)

layers = {
    "Genomics Only": genomics_df[cancer_genes],
    "Transcriptomics Only": transcriptomics_df[expr_genes],
    "Proteomics Only": proteomics_df[proteins],
    "Multi-Omics (Integrated)": X_all,
}

comparison_results = []
trained_layer_models = {}

for layer_name, X_layer in layers.items():
    X_train, X_test, y_train, y_test = train_test_split(X_layer, y, test_size=0.25, stratify=y, random_state=42)
    scaler_l = StandardScaler()
    X_train_s = scaler_l.fit_transform(X_train)
    X_test_s = scaler_l.transform(X_test)

    clf = RandomForestClassifier(n_estimators=300, random_state=42)
    clf.fit(X_train_s, y_train)
    preds = clf.predict(X_test_s)
    probs = clf.predict_proba(X_test_s)[:, 1]

    comparison_results.append({
        "Layer": layer_name,
        "Accuracy": accuracy_score(y_test, preds),
        "ROC-AUC": roc_auc_score(y_test, probs)
    })
    trained_layer_models[layer_name] = (clf, scaler_l)

comparison_df = pd.DataFrame(comparison_results)
comparison_df


,Layer,Accuracy,ROC-AUC
0,Genomics Only,0.853333,0.909286
1,Transcriptomics Only,0.893333,0.938571
2,Proteomics Only,0.880000,0.971071
3,Multi-Omics (Integrated),0.973333,0.995000


In [8]:
fig = px.bar(
    comparison_df.melt(id_vars="Layer", value_vars=["Accuracy", "ROC-AUC"]),
    x="Layer", y="value", color="variable", barmode="group",
    title="Prediction Power: Single-Omics vs Multi-Omics Integration",
    template="plotly_white", labels={"value": "Score", "variable": "Metric"},
    color_discrete_sequence=["#2E86AB", "#E63946"]
)
fig.update_layout(height=500, yaxis_range=[0.5, 1.05])
fig.show()

print("💡 Insight: Multi-Omics integration typically outperforms any single layer,")
print("   kyunke har omics layer disease biology ka ek alag pehlu capture karti hai.")


💡 Insight: Multi-Omics integration typically outperforms any single layer,
   kyunke har omics layer disease biology ka ek alag pehlu capture karti hai.


## 7. Final Multi-Omics Model — Detailed Evaluation

In [9]:
final_clf, final_scaler = trained_layer_models["Multi-Omics (Integrated)"]

X_train, X_test, y_train, y_test = train_test_split(X_all, y, test_size=0.25, stratify=y, random_state=42)
X_test_scaled = final_scaler.transform(X_test)
preds = final_clf.predict(X_test_scaled)

cm = confusion_matrix(y_test, preds)
fig = px.imshow(cm, text_auto=True, color_continuous_scale="Blues",
                 x=["No Cancer", "Cancer"], y=["No Cancer", "Cancer"],
                 labels=dict(x="Predicted", y="Actual", color="Count"),
                 title="Confusion Matrix — Multi-Omics Integrated Model")
fig.update_layout(height=450, width=500)
fig.show()

importances = pd.Series(final_clf.feature_importances_, index=feature_cols_all).sort_values(ascending=False).head(15)
layer_map = {**{g: "Genomics" for g in cancer_genes}, **{g: "Transcriptomics" for g in expr_genes}, **{p: "Proteomics" for p in proteins}}
imp_df = pd.DataFrame({"feature": importances.index, "importance": importances.values})
imp_df["layer"] = imp_df["feature"].map(layer_map)

fig2 = px.bar(imp_df, x="importance", y="feature", color="layer", orientation='h',
              title="Top 15 Predictive Features Across All Omics Layers",
              template="plotly_white", color_discrete_map={"Genomics": "#E63946", "Transcriptomics": "#2E86AB", "Proteomics": "#43AA8B"})
fig2.update_layout(height=550, yaxis={'categoryorder': 'total ascending'})
fig2.show()


## 8.  Runtime Prediction — Apna Patient Multi-Omics Data Direct Input Karein

Neeche teenon omics layers ki values daalein — model integrated features use kar ke turant **Cancer risk** predict karega.


In [10]:
# Genomics inputs (checkboxes — mutated Yes/No)
geno_boxes = {}
geno_widgets = []
for gene in cancer_genes:
    cb = widgets.Checkbox(value=False, description=f"{gene} mutated", indent=False, layout=widgets.Layout(width='200px'))
    geno_boxes[gene] = cb
    geno_widgets.append(cb)

# Transcriptomics inputs
expr_defaults = transcriptomics_df.mean().to_dict()
expr_boxes = {}
expr_widgets = []
for gene in expr_genes:
    box = widgets.FloatText(value=round(expr_defaults[gene], 2), description=gene,
                             style={'description_width': '80px'}, layout=widgets.Layout(width='200px'))
    expr_boxes[gene] = box
    expr_widgets.append(box)

# Proteomics inputs
prot_defaults = proteomics_df.mean().to_dict()
prot_boxes = {}
prot_widgets = []
for prot in proteins:
    box = widgets.FloatText(value=round(prot_defaults[prot], 2), description=prot,
                             style={'description_width': '80px'}, layout=widgets.Layout(width='200px'))
    prot_boxes[prot] = box
    prot_widgets.append(box)

predict_btn = widgets.Button(description=" Multi-Omics Predict Karein", button_style='success',
                              layout=widgets.Layout(width='280px', height='42px'))
out = widgets.Output()

def render_multiomics_result(label, proba, layer_contrib=None):
    color = "#E63946" if label == "Cancer" else "#2E86AB"
    emoji = "" if label == "Cancer" else ""
    conf = proba[1]*100 if label == "Cancer" else proba[0]*100
    html = f"""
    <div style="border:2px solid {color}; border-radius:12px; padding:18px; margin-top:12px; font-family:sans-serif; background:#fafafa;">
        <div style="font-size:22px; font-weight:700; color:{color};">{emoji} Multi-Omics Prediction: {label}</div>
        <div style="font-size:15px; margin-top:8px;">Confidence: <b>{conf:.2f}%</b></div>
        <div style="margin-top:10px; height:14px; width:100%; background:#e0e0e0; border-radius:7px; overflow:hidden;">
            <div style="height:100%; width:{proba[1]*100:.1f}%; background:linear-gradient(90deg,#2E86AB,#E63946);"></div>
        </div>
        <div style="display:flex; justify-content:space-between; font-size:12px; color:#555; margin-top:4px;">
            <span>P(No Cancer) = {proba[0]*100:.2f}%</span><span>P(Cancer) = {proba[1]*100:.2f}%</span>
        </div>
        <div style="font-size:12px; color:#777; margin-top:10px;">Based on integrated Genomics + Transcriptomics + Proteomics data</div>
    </div>
    """
    display(HTML(html))

def on_predict(b):
    with out:
        clear_output()
        row = {}
        for gene in cancer_genes:
            row[gene] = int(geno_boxes[gene].value)
        for gene in expr_genes:
            row[gene] = expr_boxes[gene].value
        for prot in proteins:
            row[prot] = prot_boxes[prot].value

        row_df = pd.DataFrame([row])[feature_cols_all]
        row_scaled = final_scaler.transform(row_df)
        pred = final_clf.predict(row_scaled)[0]
        proba = final_clf.predict_proba(row_scaled)[0]
        label = "Cancer" if pred == 1 else "No Cancer"
        render_multiomics_result(label, proba)

predict_btn.on_click(on_predict)

display(widgets.HTML("<b style='font-size:15px;'> Genomics (Mutation Status)</b>"))
display(widgets.GridBox(geno_widgets, layout=widgets.Layout(grid_template_columns="repeat(4, 210px)", grid_gap="4px")))
display(widgets.HTML("<br><b style='font-size:15px;'> Transcriptomics (Gene Expression)</b>"))
display(widgets.GridBox(expr_widgets, layout=widgets.Layout(grid_template_columns="repeat(5, 210px)", grid_gap="4px")))
display(widgets.HTML("<br><b style='font-size:15px;'> Proteomics (Protein Abundance)</b>"))
display(widgets.GridBox(prot_widgets, layout=widgets.Layout(grid_template_columns="repeat(5, 210px)", grid_gap="4px")))
display(predict_btn)
display(out)


HTML(value="<b style='font-size:15px;'> Genomics (Mutation Status)</b>")

GridBox(children=(Checkbox(value=False, description='TP53 mutated', indent=False, layout=Layout(width='200px')…

HTML(value="<br><b style='font-size:15px;'> Transcriptomics (Gene Expression)</b>")

GridBox(children=(FloatText(value=6.46, description='EXPR_1', layout=Layout(width='200px'), style=DescriptionS…

HTML(value="<br><b style='font-size:15px;'> Proteomics (Protein Abundance)</b>")

GridBox(children=(FloatText(value=4.53, description='PROT_1', layout=Layout(width='200px'), style=DescriptionS…

Button(button_style='success', description=' Multi-Omics Predict Karein', layout=Layout(height='42px', width='…

Output()